<a href="https://colab.research.google.com/github/V-Kariuki/AI_for_Science_and_Engineering/blob/develop/Week_3_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated and Explainable Machine Learning for Maternal Health Risk Prediction Across Heterogeneous Healthcare Settings

## Project problem and intended AI/ML outcome.

Healthcare facilities can serve patient populations with different characteristics and clinical-measurement patterns. A machine-learning model trained on pooled data may perform differently from models trained separately at each facility, while a single shared federated model may not perform equally well for every facility when the data are heterogeneous (non-IID).

This project will investigate maternal health risk classification by comparing **centralized learning**, **locally trained models**, and **federated learning** across simulated healthcare facilities. It will also investigate whether **personalized federated learning** can improve predictive performance for individual facilities. Explainable AI methods may be used in a later stage to help interpret model predictions.

### Main research question

> How does federated learning perform compared with centralized and locally trained machine-learning models for maternal health risk prediction when healthcare facilities have heterogeneous (non-IID) patient data, and can personalized federated learning improve prediction performance for individual healthcare facilities?

**Scope and safety note:** This notebook establishes the problem, inspects the dataset, identifies candidate features and target, and documents the proposed workflow. It does not yet train or validate a clinical prediction model. The dataset is not identified as Kenyan facility data, so findings from it must not be presented as clinically validated for Kenya. Any future use would require suitable local data, clinical review, and appropriate privacy and governance safeguards.


## 1. Data needed

The ideal study would use de-identified maternal health records from multiple healthcare facilities, with a consistent outcome definition and enough records per facility to evaluate performance. Candidate information includes maternal age, blood pressure, blood sugar, body temperature, heart rate, and a defined maternal risk label. A future, more comprehensive study could include other clinically relevant variables if they are available, lawful to use, and consistently recorded.

For this assignment, the supplied **Maternal Health Risk Data Set.csv** file is used as an initial dataset. It includes six candidate input features and a categorical `RiskLevel` label. Since the file does not provide a confirmed facility identifier in the columns shown, facility-level data will need to be simulated for methodological experiments. Simulated clients can help test non-IID scenarios, but they do not reproduce the full complexity of real healthcare facilities.


## 2. Load the dataset from Google Drive

Before running the cells, upload **Maternal Health Risk Data Set.csv** to your Google Drive. The code below mounts Drive and reads the CSV. If you save the file in a folder, update `csv_path` to match its location.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Update this path if the CSV is stored inside a folder in Google Drive.
csv_path = '/content/drive/MyDrive/Colab Notebooks/Maternal Health Risk Data Set.csv'
df = pd.read_csv(csv_path)

print(f'Dataset dimensions: {df.shape[0]} rows × {df.shape[1]} columns')
display(df.head())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset dimensions: 1014 rows × 7 columns


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.0,98.0,86,high risk
1,35,140,90,13.0,98.0,70,high risk
2,29,90,70,8.0,100.0,80,high risk
3,30,140,85,7.0,98.0,70,high risk
4,35,120,60,6.1,98.0,76,low risk


In [ ]:
# Inspect column names, data types, missing values, and basic numeric summaries.
print('Column names:')
print(df.columns.tolist())

print('\nData types:')
display(df.dtypes.rename('Data type').to_frame())

print('\nMissing values per column:')
display(df.isna().sum().rename('Missing values').to_frame())

print('\nNumeric summary:')
display(df.describe())

if 'RiskLevel' in df.columns:
    print('\nTarget-class counts:')
    display(df['RiskLevel'].value_counts().rename_axis('RiskLevel').to_frame('Number of records'))
else:
    print("The expected target column 'RiskLevel' was not found. Check the CSV column names.")


Column names:
['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel']

Data types:


,Data type
Age,int64
SystolicBP,int64
DiastolicBP,int64
BS,float64
BodyTemp,float64
HeartRate,int64
RiskLevel,object



Missing values per column:


,Missing values
Age,0
SystolicBP,0
DiastolicBP,0
BS,0
BodyTemp,0
HeartRate,0
RiskLevel,0



Numeric summary:


,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
count,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000
mean,29.871795,113.198225,76.460552,8.725986,98.665089,74.301775
std,13.474386,18.403913,13.885796,3.293532,1.371384,8.088702
min,10.000000,70.000000,49.000000,6.000000,98.000000,7.000000
25%,19.000000,100.000000,65.000000,6.900000,98.000000,70.000000
50%,26.000000,120.000000,80.000000,7.500000,98.000000,76.000000
75%,39.000000,120.000000,90.000000,8.000000,98.000000,80.000000
max,70.000000,160.000000,100.000000,19.000000,103.000000,90.000000



Target-class counts:


,Number of records
RiskLevel,
low risk,406
mid risk,336
high risk,272


Potential Model Inputs (Features) and Output (Target)
Input Features:
- Age: Age of the pregnant woman.
- SystolicBP: Systolic blood pressure.
- DiastolicBP: Diastolic blood pressure.
- BS: Blood sugar level.
- BodyTemp: Body temperature.
- HeartRate: Heart rate.
Output Target:
- RiskLevel: The maternal health risk category predicted by the model, classified as low risk, mid risk, or high risk.

## 4. Proposed machine-learning workflow

The diagram summarises the proposed workflow. This assignment covers the data-loading and problem-definition stages; the modelling stages are planned future work.

```text
       [Define research question and metrics]
                       |
                       v
           [Load CSV from Google Drive]
                       |
                       v
       [Inspect columns, units, missing data,
              labels and class balance]
                       |
                       v
         [Clean data and explore features]
                       |
                       v
       [Create reproducible train/test split]
                       |
                       v
          [Establish baseline classifiers]
                       |
                       v
     [Simulate facility clients: IID / non-IID]
                       |
                       v
  +--------------------+---------------------+
  |                    |                     |
  v                    v                     v
[Centralized]       [Local models]      [Federated model]
  |                    |                     |
  +--------------------+---------------------+
                       |
                       v
       [Evaluate overall and per-facility
       precision, recall, F1 and confusion
       matrices; examine high-risk recall]
                       |
                       v
      [Test personalized federated learning]
                       |
                       v
       [Explain predictions and compare
        performance, limitations and fairness]
```

**Brief explanation:** First, inspect and prepare the dataset and establish baseline models. Then partition the records into simulated facility clients and create controlled IID and non-IID conditions. Compare a centralized model trained on pooled training data, separate local models, and a federated model trained through model updates. Next, test a personalization method and compare results overall and for each client. Report per class and per facility metrics not accuracy alone and pay particular attention to recall for the high risk class. Explainability can help investigate model behaviour. Federated learning does not, by itself, guarantee privacy; privacy safeguards and governance must be considered separately.
